# CV04 — Trâm: Rolling Features & Dự báo Đa bước 24h

**Thành viên phụ trách:** Trâm  
**Đầu vào:** `data/processed/train.csv`, `test.csv` (CV01)  
**Đầu ra:**  
- `outputs/figures/cv04_rolling_features_impact.png`  
- `outputs/figures/cv04_multistep_forecast.png`  
- `outputs/tables/cv04_multistep_metrics.csv`  
- `data/processed/AEP_features_rolling.csv`

> **Lưu ý:** File phân tích đầy đủ nằm ở [`cv04_tram_analyst.ipynb`](cv04_tram_analyst.ipynb).  
> Notebook này là bản tóm tắt chạy được qua script `src/cv04_rolling_and_multistep.py`.

---

## 1. Cấu hình

In [ ]:
import sys, subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
FIGURES_DIR  = PROJECT_ROOT / 'outputs' / 'figures'
TABLES_DIR   = PROJECT_ROOT / 'outputs' / 'tables'
print('✅ Project root:', PROJECT_ROOT)

## 2. Chạy pipeline CV04 (qua script)

In [ ]:
script = PROJECT_ROOT / 'src' / 'cv04_rolling_and_multistep.py'
print(f'Đang chạy: {script}')
result = subprocess.run(
    [sys.executable, str(script)],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('⚠ STDERR:', result.stderr)

## 3. Xem kết quả Rolling Features Impact

In [ ]:
from IPython.display import Image
Image(str(FIGURES_DIR / 'cv04_rolling_features_impact.png'), width=900)

## 4. Xem kết quả Multi-step Forecast

In [ ]:
Image(str(FIGURES_DIR / 'cv04_multistep_forecast.png'), width=900)

## 5. Bảng chỉ số Multistep Metrics

In [ ]:
metrics_path = TABLES_DIR / 'cv04_multistep_metrics.csv'
if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print(f'Số bước dự báo: {len(df)}')
    display(df.head(10))
    display(df.tail(5))
else:
    print('⚠ Chưa có file metrics. Hãy chạy cell 2 trước.')

## 6. Nhận xét kết quả

### 6.1 Tác động của Rolling Features
- Thêm `rolling_mean_24` và `rolling_std_24` (tính trên `.shift(1).rolling(w)` — **causal**, không rò rỉ dữ liệu) cải thiện MAE và RMSE đáng kể so với chỉ dùng lag.
- `rolling_mean_168` cung cấp baseline xu hướng tuần.

### 6.2 So sánh chiến lược dự báo đa bước
| Chiến lược | Ưu điểm | Nhược điểm |
|:---|:---|:---|
| **Đệ quy** | 1 model, đơn giản | Sai số tích lũy theo bước |
| **Trực tiếp** | Không tích lũy sai số | Phải train 24 model độc lập |

### 6.3 Giải thích tại sao MAE tăng theo bước dự báo
- Trong chiến lược đệ quy, giá trị dự báo $\hat{y}_{t+1}$ được dùng làm lag đầu vào cho bước $t+2$.
- Sai số từ bước trước tích lũy vào bước sau → RMSE ở bước $h=24$ luôn cao hơn $h=1$.

## ✅ CV04 Hoàn tất

| Sản phẩm | Đường dẫn |
|:---|:---|
| Rolling Features Impact | `outputs/figures/cv04_rolling_features_impact.png` |
| Multi-step Forecast | `outputs/figures/cv04_multistep_forecast.png` |
| Multistep Metrics | `outputs/tables/cv04_multistep_metrics.csv` |
| Rolling Features Data | `data/processed/AEP_features_rolling.csv` |